<a href="https://colab.research.google.com/github/sanamm908011-a11y/project-codes/blob/main/desert_bloom_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Connect to your hard drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Re-download the Pix2Pix code (we need its brain architecture)
%cd /content/
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix
!pip install dominate fastapi python-multipart uvicorn pyngrok nest-asyncio

Mounted at /content/drive
/content
Cloning into 'pytorch-CycleGAN-and-pix2pix'...
remote: Enumerating objects: 2619, done.
remote: Total 2619 (delta 0), reused 0 (delta 0), pack-reused 2619 (from 1)
Receiving objects: 100% (2619/2619), 8.24 MiB | 19.39 MiB/s, done.
Resolving deltas: 100% (1654/1654), done.


In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import base64
import io
import sys
import cv2
import numpy as np

sys.path.insert(0, '/content/pytorch-CycleGAN-and-pix2pix')
from models.networks import define_G

print("Loading the AI...")

netG = define_G(input_nc=3, output_nc=3, ngf=64, netG='unet_256',
                norm='batch', use_dropout=False, init_type='normal', init_gain=0.02)
#netG = netG.cuda()

checkpoint_path = '/content/drive/MyDrive/desert_bloom_project/model_checkpoints/desert_bloom_experiment/latest_net_G.pth'
#state_dict = torch.load(checkpoint_path, map_location=torch.device('cuda:0'))
state_dict = torch.load(checkpoint_path, map_location=torch.device('cpu'))

netG.load_state_dict(state_dict)

netG.train()

print("✅ Model successfully loaded into the CPU!")

def translate_image(img_path):
    # 1. AI Processing
    img = Image.open(img_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((256, 256), Image.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    img_tensor = transform(img).unsqueeze(0)
    #img_tensor = transform(img).unsqueeze(0).cuda()
    with torch.no_grad():
        fake_B = netG(img_tensor)

    fake_B = (fake_B + 1) / 2.0
    img_numpy = fake_B.squeeze().cpu().float().numpy()
    img_numpy_raw = (img_numpy.transpose(1, 2, 0) * 255.0).clip(0, 255).astype('uint8')

    # 2. Math & Analysis
    img_gray = cv2.cvtColor(img_numpy_raw, cv2.COLOR_RGB2GRAY)
    total_pixels = img_gray.size
    viable_pixels = np.sum(img_gray < 140)
    score_val = int((viable_pixels / total_pixels) * 100)

    # Map to your strict TypeScript keys!
    if score_val >= 80:
        suitability_key = "preferred"
        notes = ["Excellent moisture retention detected.", "Optimal soil ratio expected.", "Uniform thermal distribution."]
    elif score_val >= 60:
        suitability_key = "good"
        notes = ["Stable thermal profile.", "Standard irrigation systems sufficient.", "Good baseline soil ratio."]
    elif score_val >= 40:
        suitability_key = "suitable"
        notes = ["Moderate thermal retention with high variance.", "Precision watering recommended for patchy zones.", "Monitor expected weed ratios."]
    else:
        suitability_key = "non-agricultural"
        notes = ["High thermal radiation detected.", "Terrain is exceptionally dry or rocky.", "Warning: Significant barren hotspots."]

    # 3. Thermal Coloring
    img_thermal = cv2.applyColorMap(img_gray, cv2.COLORMAP_INFERNO)
    img_thermal_rgb = cv2.cvtColor(img_thermal, cv2.COLOR_BGR2RGB)

    result_img = Image.fromarray(img_thermal_rgb)
    buffered = io.BytesIO()
    result_img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")

    return img_str, score_val, suitability_key, notes

Loading the AI...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/desert_bloom_project/model_checkpoints/desert_bloom_experiment/latest_net_G.pth'

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from pyngrok import ngrok
import nest_asyncio
import shutil
import random
import os

app = FastAPI()

# Allow your Vercel website to talk to Colab without security blocks
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.post("/analyze")
async def analyze_endpoint(file: UploadFile = File(...)):
    print(f"📡 Incoming transmission: {file.filename}")

    temp_file_path = f"temp_{file.filename}"
    with open(temp_file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    print("🧠 Generating Thermal Analysis...")

    # Unpack the 4 variables from our brain
    base64_img, score, suitability_key, notes = translate_image(temp_file_path)

    print(f"✅ Analysis complete. Score: {score} | Tier: {suitability_key}")

    os.remove(temp_file_path)

    # Send the perfectly formatted JSON (No GPS data here!)
    return {
        "thermal_image_base64": f"data:image/png;base64,{base64_img}",
        "score": score,
        "suitability": suitability_key,
        "notes": notes
    }

# Start the server and create a public link
nest_asyncio.apply()
ngrok.kill() # Kill any existing ngrok processes
ngrok.set_auth_token("39Oa0uk2OxD70YEjPoq7ldbfJck_FQiYnd4eAEDgwxFepxKD")
# Use your permanent domain!
public_url = ngrok.connect(8000, domain="hee-simpatico-unspeakably.ngrok-free.dev").public_url

print("=========================================================")
print(f"🚀 YOUR DESERT BLOOM API IS LIVE!")
print(f"🔗 COPY THIS URL INTO YOUR NEXT.JS APP: {public_url}")
print("=========================================================")

# THE FIX: Run the server smoothly inside Colab's existing event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

🚀 YOUR DESERT BLOOM API IS LIVE!
🔗 COPY THIS URL INTO YOUR NEXT.JS APP: https://hee-simpatico-unspeakably.ngrok-free.dev


INFO:     Started server process [2269]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
